# Генерація адаптивних шпалер — ТЕСТОВИЙ РЕЖИМ (3 варіанти)

Цей ноутбук генерує **3 тестові варіанти** перед запуском повної матриці 48 шпалер.

### Інструкція перед запуском:
1. **Додайте API-ключ у Secrets:**
   - На лівій панелі Colab натисніть на іконку ключа (**Secrets**).
   - Додайте секрет з назвою `GEMINI_API_KEY` та увімкніть **Notebook access**.
2. **Завантажте зображення:**
   - Перетягніть ваш файл кав'ярні у вкладку Files та перейменуйте його на `23820.jpg`.
3. **Запустіть комірки по порядку.**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Крок 1: Встановлення залежностей
!pip install -q google-genai pillow

In [ ]:
from google.colab import userdata
userdata.get('GEMINI_API_KEY')

In [ ]:
# Крок 2: Імпорт та ініціалізація клієнта
import os
import base64
import shutil
from google import genai
from google.genai import types
from google.colab import userdata
from IPython.display import display, Image as IPImage

try:
    api_key = userdata.get('GEMINI_API_KEY')
    client = genai.Client(api_key=api_key)
    print("✅ Клієнт успішно ініціалізовано.")
except Exception:
    print("❌ Помилка: додайте GEMINI_API_KEY у розділ Secrets та увімкніть Notebook access.")
    raise

In [ ]:
# Крок 3: Завантаження базового зображення
BASE_IMAGE_PATH = "/content/drive/MyDrive/23820.jpg"

if not os.path.exists(BASE_IMAGE_PATH):
    print(f"❌ Файл '{BASE_IMAGE_PATH}' не знайдено. Завантажте його через вкладку Files.")
else:
    # Конвертація у base64 для передачі в API
    with open(BASE_IMAGE_PATH, "rb") as f:
        image_bytes = f.read()
    image_b64 = base64.b64encode(image_bytes).decode()
    print(f"✅ Зображення завантажено ({len(image_bytes) / 1024:.1f} KB). Готово до генерації.")

In [ ]:
# Крок 4: Допоміжна функція генерації
# Використовуємо gemini-2.0-flash-preview-image-generation — модель,
# що підтримує image-to-image (вхідне зображення + текстовий промпт).

MODEL = "gemini-3.1-flash-image"

BASE_PROMPT = (
    "A neon coffee shop building on a street corner, cyberpunk aesthetic. "
    "CRITICAL: Maintain the exact architectural layout, building proportions, "
    "and the exact placement of the neon COFFEE sign from the input image. "
    "Keep the same camera angle and composition."
)

def generate_wallpaper(image_b64: str, scene_prompt: str, output_path: str) -> bool:
    """Генерує одне зображення та зберігає його у файл. Повертає True при успіху."""
    full_prompt = f"{BASE_PROMPT} Scene: {scene_prompt}"

    try:
        response = client.models.generate_content(
            model=MODEL,
            contents=[
                types.Part(
                    inline_data=types.Blob(
                        mime_type="image/jpeg",
                        data=base64.b64decode(image_b64)
                    )
                ),
                types.Part(text=full_prompt)
            ],
            config=types.GenerateContentConfig(
                response_modalities=["IMAGE", "TEXT"]
            )
        )

        # Шукаємо image-частину у відповіді
        for part in response.candidates[0].content.parts:
            if hasattr(part, 'inline_data') and part.inline_data:
                with open(output_path, "wb") as f:
                    f.write(part.inline_data.data)
                return True

        print("   ⚠️ Модель не повернула зображення. Текстова відповідь:",
              response.candidates[0].content.parts[0].text if response.candidates else "немає")
        return False

    except Exception as e:
        print(f"   ❌ Помилка API: {e}")
        return False

print("✅ Функція generate_wallpaper визначена.")

In [ ]:
# Крок 5: ТЕСТОВА ГЕНЕРАЦІЯ — 3 різних варіанти
# Мета: переконатися, що API працює та якість влаштовує перед запуском 48 зображень.

TEST_DIR = "./test_wallpapers"
os.makedirs(TEST_DIR, exist_ok=True)

TEST_CASES = [
    {
        "name": "winter_night_snow",
        "prompt": "winter, heavy snowfall, thick snow covering the roof and ground, night time, dark sky, illuminated neon coffee sign, streetlights"
    },
    {
        "name": "summer_day_clear",
        "prompt": "summer, lush green trees, clear atmosphere, midday, bright sunlight, sharp shadows, clear sky, pleasant weather"
    },
    {
        "name": "autumn_evening_rain",
        "prompt": "autumn, orange and red leaves, fallen leaves, golden hour, sunset lighting, dramatic sky, heavy rain, wet asphalt reflecting neon lights"
    },
]

print(f"🚀 Починаємо тестову генерацію ({len(TEST_CASES)} зображення)...\n")

for i, test in enumerate(TEST_CASES, 1):
    filepath = os.path.join(TEST_DIR, f"{test['name']}.jpg")
    print(f"[{i}/{len(TEST_CASES)}] Генерація: {test['name']}...")

    success = generate_wallpaper(image_b64, test["prompt"], filepath)

    if success:
        print(f"   ✅ Збережено: {filepath}")
        display(IPImage(filepath, width=640))
    else:
        print(f"   ⚠️ Пропущено: {test['name']}")

print("\n✨ Тестову генерацію завершено!")

In [ ]:
# Крок 6 (необов'язково): ПОВНА ГЕНЕРАЦІЯ — 48 варіантів
# Запускайте тільки після перевірки тестових результатів!
# Орієнтовний час: 30–60 хвилин. Орієнтовна вартість: $5–20.

OUTPUT_DIR = "./adaptive_wallpapers"
os.makedirs(OUTPUT_DIR, exist_ok=True)

SEASONS = {
    "winter": "winter, snow on the ground, bare trees",
    "spring": "spring, blooming flowers, fresh green leaves",
    "summer": "summer, lush green trees, clear atmosphere",
    "autumn": "autumn, orange and red leaves, fallen leaves"
}

TIMES_OF_DAY = {
    "morning": "early morning light, soft sunrise illumination",
    "day":     "midday, bright sunlight, sharp shadows",
    "evening": "golden hour, sunset lighting, dramatic sky",
    "night":   "night time, dark sky, illuminated neon coffee sign, streetlights"
}

WEATHER = {
    "clear":     "clear sky, pleasant weather",
    "cloudy":    "overcast sky, moody atmosphere",
    "rain_snow": "heavy rain, wet asphalt"
}

total = len(SEASONS) * len(TIMES_OF_DAY) * len(WEATHER)  # 48
current = 0
errors = []

# Імпорт для завантаження файлів
from google.colab import files

print(f"🚀 Починаємо повну генерацію ({total} зображень)...")

for season_name, season_prompt in SEASONS.items():
    for time_name, time_prompt in TIMES_OF_DAY.items():
        for weather_name, weather_prompt in WEATHER.items():
            current += 1

            # Адаптація опадів під сезон
            if weather_name == "rain_snow" and season_name == "winter":
                weather_prompt = "heavy snowfall, thick snow covering the roof and ground"
            elif weather_name == "rain_snow":
                weather_prompt = "heavy rain, wet asphalt reflecting neon lights"

            scene = f"{season_prompt}, {time_prompt}, {weather_prompt}"
            filename = f"{season_name}_{time_name}_{weather_name}.jpg"
            filepath = os.path.join(OUTPUT_DIR, filename)

            # Пропускаємо вже згенеровані файли (зручно при повторному запуску після збою)
            if os.path.exists(filepath):
                print(f"[{current}/{total}] ⏭️  Вже існує: {filename}")
                continue

            print(f"[{current}/{total}] Генерація: {season_name} | {time_name} | {weather_name}...")
            success = generate_wallpaper(image_b64, scene, filepath)

            if success:
                print(f"   ✅ Збережено: {filename}")
                files.download(filepath) # Завантажуємо файл одразу після генерації

            else:
                errors.append(filename)

# Архівування результатів
print("\n📦 Створюємо ZIP-архів...")
shutil.make_archive("wallpapers_archive", "zip", OUTPUT_DIR)
print("✅ Архів 'wallpapers_archive.zip' готовий до завантаження!")

if errors:
    print(f"\n⚠️ Не вдалося згенерувати {len(errors)} файл(ів): {errors}")
else:
    print(f"\n🎉 Всі {total} зображень згенеровано успішно!")